In [1]:
!jupyter nbextension enable --py widgetsnbextension

usage: jupyter [-h] [--version] [--config-dir] [--data-dir] [--runtime-dir]
               [--paths] [--json] [--debug]
               [subcommand]

Jupyter: Interactive Computing

positional arguments:
  subcommand     the subcommand to launch

options:
  -h, --help     show this help message and exit
  --version      show the versions of core jupyter packages and exit
  --config-dir   show Jupyter config dir
  --data-dir     show Jupyter data dir
  --runtime-dir  show Jupyter runtime dir
  --paths        show all Jupyter paths. Add --json for machine-readable
                 format.
  --json         output paths as machine-readable json
  --debug        output debug information about paths

Available subcommands: console dejavu events execute kernel kernelspec lab
labextension labhub migrate nbconvert notebook run server troubleshoot trust

Jupyter command `jupyter-nbextension` not found.


In [2]:
import torch
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, 
    TrainingArguments, Trainer, DataCollatorForLanguageModeling, BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
import re
from IPython.display import display, Markdown


In [3]:
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name())

13.0
True
NVIDIA GeForce RTX 5060 Ti


In [4]:
from tqdm.notebook import tqdm

# Загрузка модели

In [5]:
model_path = './qwen_model'
lora_path = 'qwen2.5_fine-tune/checkpoint-210'

In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_path)

In [17]:
model = AutoModelForCausalLM.from_pretrained(model_path, dtype = torch.float16, device_map = 'cuda:0')

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [18]:
torch.cuda.empty_cache()

model = PeftModel.from_pretrained(
    model,
    lora_path
)
model.eval()

c:\Users\user\projects\vkr_titova\vkr_gen_model\.venv\Lib\site-packages\peft\peft_model.py:622: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight', 'base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.1.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.1.self_attn.k_proj.lora_A.default.weight', 'base_model.model.m

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(152064, 3584)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=3584, out_features=3584, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=3584, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(

In [19]:
print(model.device)

cuda:0


________________
# Генерация ответов

## Подготовка данных

In [20]:
questions = []
file = open(r'questions.txt', 'r', encoding='utf-8')
for line in file:
    questions.append(line[:-1])
print(questions)

['Что такое вектор и чем он отличается от скаляра?', 'Дай определение производной функции.', 'Что такое электрическое сопротивление и в чём оно измеряется?', 'Что такое алгоритм? Назови его основные свойства.', 'Что такое кинетическая и потенциальная энергия?', 'Чем отличается компилятор от интерпретатора?', 'Что такое интеграл и какой физический смысл он имеет?', 'Объясни принцип работы алгоритма бинарного поиска.', 'В чём разница между упругим и неупругим ударом?', 'Что такое собственные значения матрицы и для чего они используются?', 'Объясни закон Гука и границы его применимости.', 'Что такое энтропия и как она связана со вторым началом термодинамики?', 'Чем изотермический процесс отличается от адиабатического?', 'Объясни разницу между стеком и очередью как структурами данных.', 'Что такое резонанс и при каких условиях он возникает?', 'Объясни метод наименьших квадратов.', 'В чём суть метода Рунге-Кутты и когда его применяют вместо метода Эйлера?', 'Объясни теорему Гаусса-Остроград

In [ ]:
def render_latex(text):
    '''
    функция для форматирования спец. выражений
    '''

    # убираем лишние служебные токены (если есть)
    text = text.replace("<|assistant|>", "").strip()

    # заменяем \( \) → $
    text = re.sub(r"\\\((.*?)\\\)", r"$\1$", text)

    # заменяем \[ \] → $$
    text = re.sub(r"\\\[(.*?)\\\]", r"$$\1$$", text)

    return text

In [22]:
SYSTEM_PROMPT = 'Ты — интеллектуальный помощник, специализирующийся на технических науках. Отвечай ТОЛЬКО на русском языке, максимально точно и с объяснением. Для генерации специальных символов и формул используй ТОЛЬКО формат LaTeX. Прописывай названия уравнений, формул и законов, если это используется в решении.'

In [23]:
def generate(task, sys = SYSTEM_PROMPT):
    messages = [
        {"role": "system", "content": sys}, 
        {"role": "user", "content": task}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=2048)
        generated_tokens = output[0][inputs["input_ids"].shape[-1]:]


    formatted = render_latex(tokenizer.decode(generated_tokens, skip_special_tokens=True))

    return formatted

In [24]:
for el in tqdm(questions, desc='Идёт генерация ответов..'):
    display(Markdown(f'Вопрос: {el}'))
    display(Markdown(f'Ответ: {generate(el)}'))
    display(Markdown('______'))

Идёт генерация ответов..:   0%|          | 0/30 [00:00<?, ?it/s]

Вопрос: Что такое вектор и чем он отличается от скаляра?

Ответ: Вектор и скаляр — это два типа физических величин, которые используются в физике и математике для описания различных свойств объектов.

Скаляр — это физическая величина, которая полностью определяется своим значением (модулем) без указания направления. Например, масса, температура или время являются скалярами. Скаляры можно складывать и вычитать как обычные числа.

Вектор же — это физическая величина, которая характеризуется не только значением (модулем), но и направлением в пространстве. Векторы обозначаются жирным шрифтом или стрелкой над буквой, например, $\mathbf{A}$. Основные операции с векторами включают сложение, вычитание и умножение на скаляр.

Основные отличия между скаляром и вектором:

1. **Модуль и направление**:
   - Скаляр: имеет только модуль (значение).
   - Вектор: имеет модуль и направление.

2. **Арифметические операции**:
   - Скаляр: можно складывать, вычитать, умножать и делить.
   - Вектор: можно складывать, вычитать и умножать на скаляр, но деление на вектор не определено.

3. **Геометрическое представление**:
   - Скаляр: представлен просто числом.
   - Вектор: геометрически представляется как направленный отрезок, где длина отрезка соответствует модулю вектора, а направление — направлению вектора.

Примеры векторных величин в физике включают скорость, ускорение, силу и импульс. Эти величины не могут быть полностью описаны без указания их направления.

______

Вопрос: Дай определение производной функции.

...

Ответ: Для решения этой задачи будем использовать свойство алгоритма, который выполняется за время $O(n^2)$. Это означает, что время выполнения $T(n)$ можно представить в виде:

$$ T(n) = c \cdot n^2 $$

где $c$ — это некоторая константа.

По условию, при $n = 100$ алгоритм работает 1 секунду:

$$ T(100) = c \cdot 100^2 = 1 \text{ секунда} $$

Отсюда мы можем выразить константу $c$:

$$ c = \frac{1}{100^2} = \frac{1}{10000} $$

Теперь, чтобы найти время выполнения при $n = 1000$, подставим $n = 1000$ в формулу:

$$ T(1000) = c \cdot 1000^2 = \frac{1}{10000} \cdot 1000^2 = \frac{1000^2}{10000} = \frac{1000000}{10000} = 100 \text{ секунд} $$

Таким образом, при $n = 1000$ алгоритм будет работать 100 секунд.

______

Вопрос: Тело брошено горизонтально со скоростью 10 м/с с высоты 20 м. Найди дальность полёта

Ответ: Для решения задачи о движении тела, брошенного горизонтально, мы будем использовать законы механики. В данном случае важно понимать, что вертикальное и горизонтальное движения происходят независимо друг от друга.

### Вертикальное движение

1. **Начальная скорость по оси $ y $:** $ v_{0y} = 0 $ (так как тело брошено горизонтально).
2. **Ускорение по оси $ y $:** $ g = 9.8 \, \text{м/с}^2 $ (ускорение свободного падения).

Используем уравнение движения для определения времени падения:
\[
h = v_{0y} t + \frac{1}{2} g t^2
\]
Подставляем значения:
\[
20 = 0 \cdot t + \frac{1}{2} \cdot 9.8 \cdot t^2
\]
\[
20 = 4.9 t^2
\]
\[
t^2 = \frac{20}{4.9}
\]
\[
t^2 \approx 4.08
\]
\[
t \approx \sqrt{4.08} \approx 2.02 \, \text{с}
\]

### Горизонтальное движение

1. **Начальная скорость по оси $ x $:** $ v_{0x} = 10 \, \text{м/с} $ (дано в условии).
2. **Ускорение по оси $ x $:** $ a_x = 0 $ (горизонтальное движение без ускорения).

Используем уравнение движения для определения горизонтальной дальности:
\[
x = v_{0x} t
\]
Подставляем значения:
\[
x = 10 \cdot 2.02 \approx 20.2 \, \text{м}
\]

Таким образом, дальность полета тела составляет около $ \boxed{20.2 \, \text{м}} $.

______